# Week 7 — Day 4: Custom Dataset Class — Intel Image Classification

## Goal
Build custom Dataset class, fine-tune EfficientNet-B0 on Intel dataset.
Target: above 90% accuracy.
Result: 93.27% ✅

---

## Dataset — Intel Image Classification
6 classes: buildings, forest, glacier, mountain, sea, street
14,034 train images, 3,000 test images
Source: kaggle.com/datasets/puneet6060/intel-image-classification

---

## Why Custom Dataset Class?
Built-in datasets (CIFAR10, MNIST) only work for pre-packaged data.
Real projects have images in custom folders with custom labels.
Custom Dataset class loads ANY data from ANY folder structure.

Every custom Dataset needs 3 methods:

| Method | Purpose |
|---|---|
| __init__ | Load file paths and labels into memory |
| __len__ | Return total number of samples — DataLoader needs this |
| __getitem__ | Return one image+label at index idx — DataLoader calls this |

---

## Custom Dataset Implementation

```python
class IntelDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.images    = []
        self.labels    = []
        self.classes   = sorted(os.listdir(root_dir))
        
        for label, cls in enumerate(self.classes):
            cls_path = os.path.join(root_dir, cls)
            for img_file in os.listdir(cls_path):
                self.images.append(os.path.join(cls_path, img_file))
                self.labels.append(label)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label    = self.labels[idx]
        image    = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label
```

### Line by line — __init__:
- `sorted(os.listdir(root_dir))` — gets class folder names alphabetically
- `enumerate(self.classes)` — gives label 0,1,2... for each class
- Loops through every image in every class folder
- Stores full path and numeric label for each image

### Line by line — __getitem__:
- `self.images[idx]` — path of image at index idx
- `Image.open().convert('RGB')` — opens image, converts to RGB
  (some images are grayscale or RGBA — convert ensures 3 channels)
- `self.transform(image)` — applies augmentation/normalization
- Returns image tensor and label integer

### Common bug — nested folders:
Intel dataset extracts as seg_train/seg_train/ — extra nested level.
Always check actual path before creating Dataset:
```python
# wrong
IntelDataset('/content/intel/seg_train')
# correct
IntelDataset('/content/intel/seg_train/seg_train')
```

---

## Model Setup

```python
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# freeze all layers
for param in model.parameters():
    param.requires_grad = False

# replace for 6 classes (not 10 like CIFAR-10)
model.classifier[1] = nn.Linear(1280, 6)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
```

**Trainable parameters: 7,686** — only last layer

---

## Training Results

### Stage 1 — Frozen (10 epochs)
| Epoch | Train Acc | Test Acc |
|---|---|---|
| 1 | 83.07% | 88.03% |
| 5 | 88.18% | 89.90% |
| 10 | 88.79% | 89.57% |

**Best frozen accuracy: 90.23%**

### Stage 2 — Fine-tuning (5 epochs)
```python
for param in model.parameters():
    param.requires_grad = True
optimizer = optim.Adam(model.parameters(), lr=1e-4)
```

| Epoch | Train Acc | Test Acc |
|---|---|---|
| 1 | 91.25% | 92.57% |
| 3 | 96.37% | 92.63% |
| 5 | 98.16% | 93.03% |

**Best fine-tune accuracy: 93.27% ✅**

---

## Save and Load Model

```python
# save
torch.save(model.state_dict(), 'intel_efficientnet.pth')

# load
model_loaded = models.efficientnet_b0(weights=None)
model_loaded.classifier[1] = nn.Linear(1280, 6)
model_loaded.load_state_dict(torch.load('intel_efficientnet.pth'))
model_loaded = model_loaded.to(device)
model_loaded.eval()
```

**state_dict()** — saves only weights, not architecture.
Must recreate architecture first, then load weights into it.

---

## Inference on 5 Test Images

```python
output.argmax(dim=1)           # predicted class
torch.softmax(output, dim=1).max()  # confidence score
```

**Results:**
| True | Predicted | Confidence | Result |
|---|---|---|---|
| sea | sea | 100.00% | ✅ |
| buildings | buildings | 99.99% | ✅ |
| sea | sea | 100.00% | ✅ |
| sea | sea | 100.00% | ✅ |
| glacier | sea | 99.26% | ❌ |

**Key observation — overconfident wrong prediction:**
Glacier predicted as sea with 99.26% confidence.
Both have blue/white colors and similar textures.
In production this is dangerous — model says 99% sure but wrong.
Solution: temperature scaling or ensemble to calibrate confidence.

---

## Full Pipeline Summary


---

## Key Takeaways
- Custom Dataset needs __init__, __len__, __getitem__ — always
- convert('RGB') in __getitem__ — handles grayscale and RGBA images
- Always check folder structure before writing Dataset path
- 6 classes → nn.Linear(1280, 6) not 1280→10
- state_dict() saves weights only — must rebuild architecture to load
- High confidence wrong predictions are dangerous in production
- Fine-tuning always beats frozen training — two stage approach works best
- Next: Day 5 — Grad-CAM, visualize what CNN sees

# Custom Dataset Class :

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

class IntelDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.images    = []
        self.labels    = []
        self.classes   = sorted(os.listdir(root_dir))

        for label, cls in enumerate(self.classes):
            cls_path = os.path.join(root_dir, cls)
            for img_file in os.listdir(cls_path):
                self.images.append(os.path.join(cls_path, img_file))
                self.labels.append(label)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label    = self.labels[idx]
        image    = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# ── outside class ──────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dataset = IntelDataset('/content/intel/seg_train/seg_train', transform=train_transform)
test_dataset  = IntelDataset('/content/intel/seg_test/seg_test',  transform=val_transform)

print(f"Classes: {train_dataset.classes}")
print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")

image, label = train_dataset[0]
print(f"Image shape: {image.shape}")
print(f"Label: {label} — {train_dataset.classes[label]}")

Classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Train size: 14034
Test size: 3000
Image shape: torch.Size([3, 224, 224])
Label: 0 — buildings


# Create DataLoader and Build The Model :

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32,
                          shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32,
                          shuffle=False, num_workers=2)

device = torch.device('cuda')

# load EfficientNet-B0
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
print(model.classifier)

# freeze all layers
# freeze all layers
for param in model.parameters():
    param.requires_grad = False

# replace classifier for 6 classes
model.classifier[1] = nn.Linear(1280, 6)

# move to devices
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

print(model.classifier)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=6, bias=True)
)
Trainable parameters: 7,686


# Training Loop :

In [15]:
train_accs = []
test_accs  = []
best_acc   = 0

for epoch in range(10):
    # training phase
    model.train()
    correct = 0
    total   = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        # forward + backward + step
        optimizer.zero_grad()
        output = model(images)
        loss   = criterion(output, labels)
        loss.backward()
        optimizer.step()

        # count correct predictions
        pred     = output.argmax(dim=1)  # highest score = predicted digit
        correct += (pred == labels).sum().item()
        total   += labels.size(0)

    train_acc = correct / total

    # validation phase
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            # predict and count correct
            output   = model(images)
            pred     = output.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total   += labels.size(0)

    test_acc = correct / total
    if test_acc > best_acc:
        best_acc = test_acc

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(f"Epoch {epoch+1:2d} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Test Acc: {test_acc:.4f}")

print(f"\nBest Test Accuracy: {best_acc:.4f}")

Epoch  1 | Train Acc: 0.8307 | Test Acc: 0.8803
Epoch  2 | Train Acc: 0.8739 | Test Acc: 0.8900
Epoch  3 | Train Acc: 0.8824 | Test Acc: 0.8903
Epoch  4 | Train Acc: 0.8829 | Test Acc: 0.8943
Epoch  5 | Train Acc: 0.8818 | Test Acc: 0.8990
Epoch  6 | Train Acc: 0.8851 | Test Acc: 0.8960
Epoch  7 | Train Acc: 0.8856 | Test Acc: 0.8930
Epoch  8 | Train Acc: 0.8866 | Test Acc: 0.9023
Epoch  9 | Train Acc: 0.8853 | Test Acc: 0.8957
Epoch 10 | Train Acc: 0.8879 | Test Acc: 0.8957

Best Test Accuracy: 0.9023


# Fine-tuning: unfreeze all layers and train with very low LR:

In [16]:
 # unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-4)

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Trainable parameters: 4,015,234


# Fine Tuning :

In [17]:
train_accs_ft = []
test_accs_ft  = []

for epoch in range(5):
    model.train()
    correct = 0
    total   = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(images)
        loss   = criterion(output, labels)
        loss.backward()
        optimizer.step()

        pred     = output.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total   += labels.size(0)

    train_acc = correct / total

    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            output   = model(images)
            pred     = output.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total   += labels.size(0)

    test_acc = correct / total
    train_accs_ft.append(train_acc)
    test_accs_ft.append(test_acc)

    print(f"Fine-tune Epoch {epoch+1} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Test Acc: {test_acc:.4f}")

print(f"\nBest fine-tune accuracy: {max(test_accs_ft):.4f}")

Fine-tune Epoch 1 | Train Acc: 0.9125 | Test Acc: 0.9257
Fine-tune Epoch 2 | Train Acc: 0.9513 | Test Acc: 0.9323
Fine-tune Epoch 3 | Train Acc: 0.9637 | Test Acc: 0.9263
Fine-tune Epoch 4 | Train Acc: 0.9778 | Test Acc: 0.9327
Fine-tune Epoch 5 | Train Acc: 0.9816 | Test Acc: 0.9303

Best fine-tune accuracy: 0.9327


# Save Model :

In [18]:
# save model
torch.save(model.state_dict(), 'intel_efficientnet.pth')
print("Model saved!")

# load model back
model_loaded = models.efficientnet_b0(weights=None)
model_loaded.classifier[1] = nn.Linear(1280, 6)
model_loaded.load_state_dict(torch.load('intel_efficientnet.pth'))
model_loaded = model_loaded.to(device)
model_loaded.eval()
print("Model loaded!")

Model saved!
Model loaded!


# Run Infrence :

In [19]:
classes = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

# get 5 random test images
import random
indices = random.sample(range(len(test_dataset)), 5)

for idx in indices:
    image, true_label = test_dataset[idx]
    image_tensor = image.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model_loaded(image_tensor)
        pred   = output.argmax(dim=1).item()
        conf   = torch.softmax(output, dim=1).max().item()

    print(f"True: {classes[true_label]:12s} | "
          f"Predicted: {classes[pred]:12s} | "
          f"Confidence: {conf:.2%} | "
          f"{'✅' if pred == true_label else '❌'}")

True: sea          | Predicted: sea          | Confidence: 100.00% | ✅
True: buildings    | Predicted: buildings    | Confidence: 99.99% | ✅
True: sea          | Predicted: sea          | Confidence: 100.00% | ✅
True: sea          | Predicted: sea          | Confidence: 100.00% | ✅
True: glacier      | Predicted: sea          | Confidence: 99.26% | ❌
